<a href="https://colab.research.google.com/github/Zelenenene/git-and-github-final-project-Zelenenene/blob/main/MSE641_Task_1_RoBERTa_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
import pandas as pd
import numpy as np
import torch
import random

from transformers import AutoTokenizer
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from sklearn.metrics import f1_score
from transformers import Trainer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from transformers import set_seed

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


#Check the label distribution in the training set
train = pd.read_json("train.jsonl", lines=True)

print(train["tags"].value_counts())

tags
[phrase]     1367
[passage]    1274
[multi]       559
Name: count, dtype: int64


See that we have three types of tags here, "phrase", "passage", or "multi". Conforming to the descriptions in Task 1.

In [ ]:
#See the first few rows for the training set

train.head()

,uuid,postId,postText,postPlatform,targetParagraphs,targetTitle,targetDescription,targetKeywords,targetMedia,targetUrl,provenance,spoiler,spoilerPositions,tags
0,0af11f6b-c889-4520-9372-66ba25cb7657,532quh,"[Wes Welker Wanted Dinner With Tom Brady, But ...",reddit,[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"new england patriots, ricky doyle, top stories,","[http://pixel.wp.com/b.gif?v=noscript, http://...",http://nesn.com/2016/09/wes-welker-wanted-dinn...,"{'source': 'anonymized', 'humanSpoiler': 'They...",[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,411701128456593408,[NASA sets date for full recovery of ozone hole],Twitter,[2070 is shaping up to be a great year for Mot...,Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,"ozone layer,ozone hole determined by weather,M...",[http://s.m.huffpost.com/assets/Logo_Huffingto...,http://huff.to/1cH672Z,"{'source': 'anonymized', 'humanSpoiler': '2070...",[2070],"[[[0, 0], [0, 4]]]",[phrase]
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,380537005123190784,[This is what makes employees happy -- and it'...,Twitter,"[Despite common belief, money isn't the key to...",Intellectual Stimulation Trumps Money For Empl...,By: Chad Brooks \r\nPublished: 09/18/2013 06:4...,"employee happiness money,employee happiness in...",[http://i.huffpost.com/gen/1359674/images/o-HA...,http://huff.to/1epfeaw,"{'source': 'anonymized', 'humanSpoiler': 'Inte...",[intellectual stimulation],"[[[1, 186], [1, 210]]]",[phrase]
3,31ecf93c-3e21-4c80-949b-aa549a046b93,844567852531286016,[Passion is overrated — 7 work habits you need...,Twitter,"[It’s common wisdom. Near gospel really, and n...","‘Follow your passion’ is wrong, here are 7 hab...",There's a lot more to work that loving your job,"business, work-life, careers",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[Purpose connects us to something bigger and i...,"[[[11, 25], [11, 101]], [[17, 56], [17, 85]], ...",[multi]
4,31b108a3-c828-421a-a4b9-cf651e9ac859,814186311573766144,[The perfect way to cook rice so that it's per...,Twitter,"[Boiling rice may seem simple, but there is a ...",Revealed: The perfect way to cook rice so that...,The question 'How does one cook rice properly?...,"Quora,users,share,perfect,way,cook,rice",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[in a rice cooker],"[[[5, 60], [5, 76]]]",[phrase]


In [ ]:
#Also, see the column names of the train dataset
train.columns

Index(['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs',
       'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia',
       'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags'],
      dtype='object')

Based on the dataset description, 14 variables are available. The target variable is tags, which represents three spoiler categories: phrase, passage, and multi.

In [ ]:
#Read the test dataset as well as the validation set
test = pd.read_json("test.jsonl", lines=True)

val = pd.read_json("val.jsonl", lines=True)

#See what variables are available in the test dataset
test.columns

Index(['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle',
       'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl',
       'id'],
      dtype='object')

For the input variables, we mainly focus on postText, targetTitle, and targetParagraphs. The post text captures the linguistic characteristics of clickbait expressions, while the webpage title and paragraphs provide contextual information about the underlying content and spoiler type. These textual features are then transformed into model inputs through tokenization.

In [ ]:
#Convert the three predictors into the input of the model.
def create_text(df):

    df["paragraph_text"] = df["targetParagraphs"].apply(
        lambda x: " ".join(x)
    )

    df["post_text"] = df["postText"].apply(
    lambda x: " ".join(x)
)

    df["text"] = (
        df["post_text"]
        + " "
        + df["targetTitle"]
        + " "
        + df["paragraph_text"]
    )

    return df

train_df = create_text(train)
val_df = create_text(val)
test_df = create_text(test)

train_df["text"].head()

,text
0,"Wes Welker Wanted Dinner With Tom Brady, But P..."
1,NASA sets date for full recovery of ozone hole...
2,This is what makes employees happy -- and it's...
3,Passion is overrated — 7 work habits you need ...
4,The perfect way to cook rice so that it's perf...


In [ ]:
#Then, we would like to deal with the target variable, mapping them to 0, 1, and 2
label_mapping = {
    "phrase":0,
    "passage":1,
    "multi":2
}

train_df["label"] = train_df["tags"].apply(
    lambda x: label_mapping[x[0]]
)

val_df["label"] = val_df["tags"].apply(
    lambda x: label_mapping[x[0]]
)

Based on the distribution of tags, the three spoiler categories are relatively imbalanced. We analyze the text length distribution and examine the token length after applying the tokenizer.

In [ ]:
train_df["text"].apply(
    lambda x: len(x.split())
).describe()

,text
count,3200.000000
mean,536.808437
std,583.082046
min,17.000000
25%,244.750000
50%,373.500000
75%,661.250000
max,14078.000000


In [ ]:
#We use the Roberta model to check tokenizer
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df["text"].apply(
    lambda x: len(tokenizer.encode(x))
).describe()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2585 > 512). Running this sequence through the model will result in indexing errors


,text
count,3200.000000
mean,697.498750
std,906.913706
min,20.000000
25%,319.000000
50%,490.000000
75%,839.250000
max,33566.000000


The token length analysis shows that some input sequences exceed the maximum sequence length of 512 tokens. Therefore, the original text cannot be directly used as input without further preprocessing.

To address this issue, we apply sequence length handling strategies before model training.

In [ ]:
#We can reduce the number of paragraphs, redefine the text input.
#still use the first 5 in this case.
def create_text_new(df):

    df["paragraph_text"] = df["targetParagraphs"].apply(
    lambda x: " ".join(x[:5])
)

    df["post_text"] = df["postText"].apply(
    lambda x: " ".join(x)
)

    df["text"] = (
        "Post: "
        + df["post_text"]
        + " Title: "
        + df["targetTitle"]
        + " Article: "
        + df["paragraph_text"]
    )

    return df

train_df_new = create_text_new(train)
val_df_new = create_text_new(val)
test_df_new = create_text_new(test)

lengths = train_df_new["text"].apply(
    lambda x: len(tokenizer.encode(x))
)

lengths.describe()

,text
count,3200.000000
mean,275.180000
std,118.357864
min,26.000000
25%,198.750000
50%,257.000000
75%,326.000000
max,1432.000000


Based on the results, this version of text preprocessing is more suitable for Roberta compared with the previous approach. However, we still need to evaluate the proportion of input sequences that exceed the maximum length limit of 512 tokens.

In [ ]:
#Count how many are over 512.
(lengths > 512).sum()

np.int64(117)

In [ ]:
#Count the percentage of them over 512.
(lengths > 512).mean()

np.float64(0.0365625)

This means that about 4% of the samples exceed 512 tokens. Approximately 96% were within the limit. This means that the vast majority of texts will not be truncated and only a small number of long texts will lose the latter part.

Then, the tokenizer applies truncation to longer sequences and padding to shorter sequences, ensuring that all inputs have a fixed length compatible with the model.

In [ ]:
train_encodings = tokenizer(
    train_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

val_encodings = tokenizer(
    val_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

test_encodings = tokenizer(
    test_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

In [ ]:
#Create Dataset
class ClickbaitDataset(Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])

        return item

After tokenization, the data are organized into training, validation, and test datasets. The training and validation datasets contain input features and labels, while the test dataset only contains input features because the true labels are not provided for evaluation.

In [ ]:
#Notice that no 'label' for test dataset
train_dataset = ClickbaitDataset(
    train_encodings,
    train_df_new["label"].tolist()
)

val_dataset = ClickbaitDataset(
    val_encodings,
    val_df_new["label"].tolist()
)

test_dataset = ClickbaitDataset(
    test_encodings
)

Then, load the classification model

In [ ]:
num_labels = 3 #As we have three classes

#Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


After that, the training parameters are configured, including the learning rate, batch size, number of training epochs, and weight decay. These parameters control the optimization process and affect the model's performance during fine-tuning.

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_input_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="weighted_f1",

    greater_is_better=True
)

We fine-tuned a pretrained model for spoiler type classification. The model was trained for 3 epochs with a learning rate of 2e-5 and batch size of 8. The maximum input sequence length was set to 512 tokens. Validation performance was monitored after each epoch using weighted F1-score, and the best-performing checkpoint was selected.

Since the competition evaluates performance using F1-score, weighted F1-score was used to monitor validation performance after each epoch, and the best-performing checkpoint was selected based on this metric.

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "weighted_f1": f1
    }

Then we set up the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

Then, the configured Trainer is applied to the training dataset to fine-tune the Roberta model.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1
1,No log,0.774954,0.684385
2,0.915781,0.688777,0.718876
3,0.645042,0.707094,0.763786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.7341192690531413, metrics={'train_runtime': 967.5414, 'train_samples_per_second': 9.922, 'train_steps_per_second': 1.24, 'total_flos': 2525888810188800.0, 'train_loss': 0.7341192690531413, 'epoch': 3.0})

The fine-tuned model is evaluated on the validation dataset using the Trainer's evaluation function.

In [ ]:
#See the evaluate score
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Weighted F1
0.645042,0.707094,3,0.763786


{'eval_loss': 0.7070944309234619, 'eval_weighted_f1': 0.7637862745880685}

To further analyze the classification performance, predictions were generated on the validation dataset. The model outputs logits for each class, and the class with the highest probability was selected as the predicted label using the argmax function. The predicted labels were then compared with the reference validation labels for performance evaluation.

In [ ]:
val_output = trainer.predict(val_dataset)

val_logits = val_output.predictions

val_pred = val_logits.argmax(axis=1)

val_labels = val_df_new["label"].values

In [ ]:
print(
    classification_report(
        val_labels,
        val_pred,
        target_names=[
            "phrase",
            "passage",
            "multi"
        ]
    )
)

              precision    recall  f1-score   support

      phrase       0.76      0.78      0.77       162
     passage       0.75      0.81      0.78       154
       multi       0.83      0.64      0.72        84

    accuracy                           0.77       400
   macro avg       0.78      0.75      0.76       400
weighted avg       0.77      0.77      0.76       400



Then we can produce a confusion matrix for the validation set.

In [ ]:
cm = confusion_matrix(
    val_labels,
    val_pred
)

pd.DataFrame(
    cm,
    index=["phrase","passage","multi"],
    columns=["phrase","passage","multi"]
)

,phrase,passage,multi
phrase,127,30,5
passage,23,125,6
multi,18,12,54


After the validation process, the model is applied to the test dataset to generate final predictions. Since the test labels are not provided, the model outputs are used directly to determine the predicted spoiler types for each test sample.

In [ ]:
test_output = trainer.predict(test_dataset)

test_pred = test_output.predictions.argmax(axis=1)

#Use the label we have for mapping
id_to_label = {
    0:"phrase",
    1:"passage",
    2:"multi"
}

test_types = [
    id_to_label[x]
    for x in test_pred
]

In [ ]:
#Then we have the submission
submission_task1 = pd.DataFrame({
    "id": test_df["id"],
    "spoilerType": test_types
})

#Check the head of the submission
submission_task1.head()

,id,spoilerType
0,0,multi
1,1,passage
2,2,phrase
3,3,phrase
4,4,phrase


Finally, the predicted spoiler types are saved as a CSV file. This file contains the predicted labels for all test samples and is submitted for evaluation.

In [ ]:
submission_task1.to_csv(
    "prediction_task1.csv",
    index=False
)